In [ ]:
import sys
import os
import torch
from torch import Tensor
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial
import einops
from rich import print as rprint
from rich.table import Table
from typing import List
from jaxtyping import Float
import numpy as np
import seaborn as sns

import transformer_lens
from transformer_lens import HookedTransformer
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import FactoredMatrix
import transformer_lens.utils as utils

torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

%load_ext autoreload
%autoreload 2

In [ ]:
# flags for simplyfying the model internals
model = HookedTransformer.from_pretrained(
    "meta-llama/Llama-3.2-1B",   
    # 'pythia-14m', # rope_freq = 10000
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=False,
    default_prepend_bos=False
)
device: torch.device = utils.get_device()

model = model.to(device)    

In [ ]:
from data.succession import generate_successor_pairs, create_prompt, create_flipped_prompt, create_augmented_prompts

succession_dataset, succession_mapping = generate_successor_pairs()

task_prompts = create_prompt(succession_dataset)
flipped_task_prompts = create_flipped_prompt(succession_dataset)

aug_data = create_augmented_prompts(task_prompts, flipped_task_prompts)

create_single_prompt_lambda = lambda task_prompts: " ".join(task_prompts.values())
prompts = create_single_prompt_lambda(task_prompts)

for task_name, task_prompt in task_prompts.items():
    print(f"{task_name}: {task_prompt}")

aug_clean_prompts = [aug["clean"] for aug in aug_data]
aug_flipped_prompts = [aug["corrupt"] for aug in aug_data]
for i, (task_name, task_prompt) in enumerate(zip(task_prompts, flipped_task_prompts)):
    aug_prompt = aug_data[i]["clean"]

## Effective Embedding

Introduced in https://arxiv.org/pdf/2310.04625. Appendix I provides a definition for the `effective embedding`: 

"Prior research on GPT-2 Small has found the counter-intuitive result that at the stage of a circuit where the input token’s value is needed, the output of MLP0 is often more important for token predictions than the model’s embedding layer (Wang et al., 2023; Hanna et al., 2023). To account for this, we define the effective embedding. The effective embedding is purely a function of the input token, with no leakage from other tokens in the prompt, as the attention is ablated. Why choose to extend the embedding up to MLP0 rather than another component in the model? This is because if we run forward passes with GPT-2 Small where we delete WE from the residual stream just after MLP0 has been added to the residual stream, cross entropy loss decreases"

The effective embedding is applied by summing the output of `MLP0` to the residual stream after Attention (resid_tokens (`resid_pre`) + `attn_out`), i.e. `resid_mid`, so the full operation of the effective embedding is given by `W_EE_full = resid_pre` + `attn_out` + `mlp_out`. 

The effective embedding can be used when we subsequently analyze independent weighted parameters of the Transformer, i.e. for Attention Heads we can take the Attention Output (or the QK/OV circuits) and project them onto this effective embedding space. If `attn_out` is of shape (d_head, d_model) and `W_EE_full @ W_U` is of shape (d_vocab, d_model) then in order to project onto the effective embedding space we need to multiply `attn_out` with the emdedding (W_U is the transposed of W_E if they are tied) in order to have an outer dimension of `d_vocab` such that by multiplying with the previous matrix results in a low-rank matrix of shape `d_vocab, d_vocab`. We can then inspect the elements across the diagonal to see the proportion and sort them by their respective value (in logits).

In [ ]:
# # code taken from https://github.com/callummcdougall/SERI-MATS-2023-Streamlit-pages/blob/main/transformer_lens/rs/callum2/ov_qk_circuits/section_41_42.ipynb
# # TODO: This OOMS for Llama 3.2-1B on 8GB, doesn't on GPT-2 Small
# def get_effective_embedding(model: HookedTransformer, use_codys_without_attention_changes=True) -> Float[Tensor, "d_vocab d_model"]:

#     W_E = model.W_E
#     W_U = model.W_U
#     # t.testing.assert_close(W_E[:10, :10], W_U[:10, :10].T)  NOT TRUE, because of the center unembed part!
    
#     resid_pre = W_E.unsqueeze(0)

#     if not use_codys_without_attention_changes:
#         pre_attention = model.blocks[0].ln1(resid_pre)
#         attn_out = einops.einsum(
#             pre_attention, 
#             model.W_V[0],
#             model.W_O[0],
#             "b s d_model, num_heads d_model d_head, num_heads d_head d_model_out -> b s d_model_out",
#         )
#         resid_mid = attn_out + resid_pre
#         del pre_attention, attn_out
#     else:
#         resid_mid = resid_pre

#     normalized_resid_mid = model.blocks[0].ln2(resid_mid)
#     mlp_out = model.blocks[0].mlp(normalized_resid_mid)
    
#     W_EE = mlp_out.squeeze()
#     W_EE_full = resid_mid.squeeze() + mlp_out.squeeze()

#     del resid_pre, resid_mid, normalized_resid_mid, mlp_out
#     torch.cuda.empty_cache()

#     return {
#         "W_E (no MLPs)": W_E,
#         "W_E (including MLPs)": W_EE_full,
#         "W_E (only MLPs)": W_EE,
#         "W_U": W_U.T,
#     }

# effective_embedding_dict = get_effective_embedding(model, use_codys_without_attention_changes=False)
# W_EE = effective_embedding_dict["W_E (including MLPs)"]
# W_EE0 = effective_embedding_dict["W_E (only MLPs)"]
# W_U = model.W_U

# print(W_EE.shape, W_EE0.shape, W_U.shape)

In [ ]:
# from utils.plot_head import hist

# # Because previous cell OOMs, we keep this commented
# W_V = model.W_V[10, 7]
# W_O = model.W_O[10, 7]

# # full_OV_circuit = (W_EE @ (W_V @ W_O) @ W_U)
# full_OV_circuit = FactoredMatrix(W_EE @ W_V, W_O @ W_U)
# full_OV_circuit.shape

# diag_elem_negative_ranks = []

# for i in tqdm(range(model.cfg.d_vocab)):
#     col = full_OV_circuit.A[i, :] @ full_OV_circuit.B
#     diag_elem = col[i]
#     diag_elem_negative_rank = (diag_elem > col).sum().item() # This is zero when diag elem is minimal
#     diag_elem_negative_ranks.append(diag_elem_negative_rank)

# diag_elem_negative_ranks = torch.tensor(diag_elem_negative_ranks)

# print(f"Mean rank = {diag_elem_negative_ranks.float().mean().item():.2f}")
# print(f"Median rank = {diag_elem_negative_ranks.float().median().item():.0f}\n")

# print(f"Proportion with rank zero = {(diag_elem_negative_ranks == 0).float().mean():.2%}")
# print(f"Proportion with rank less than 10 = {(diag_elem_negative_ranks < 10).float().mean():.2%}\n")

# print(f"Quantity with rank more than 10% = {(diag_elem_negative_ranks > 0.1 * model.cfg.d_vocab).sum()}")
# print(f"Quantity with rank more than 5% = {(diag_elem_negative_ranks > 0.05 * model.cfg.d_vocab).sum()}")


# def create_title_and_subtitles(
#     title: str,
#     subtitles: List[str],
# ) -> str:
#     return f"{title}<br><span style='font-size:13px'>{'<br>'.join(subtitles)}</span>"

# # Histogram, with all values above 100 cropped so the pattern is more visible
# hist(
#     diag_elem_negative_ranks[diag_elem_negative_ranks < 100], 
#     title=create_title_and_subtitles("Dynamic analysis: suppression ranks of source tokens", ["Rank=0 means source token is the most suppressed"]),
#     template="simple_white",
#     labels={"x": "Rank"},
# )

In [ ]:
# # Because previous cell OOMs, we keep this commented
# tokens_not_in_top_10pct = torch.arange(model.cfg.d_vocab)[diag_elem_negative_ranks > 0.1 * model.cfg.d_vocab]
# print(f"Number of tokens not in top 5% = {len(tokens_not_in_top_10pct)}")

# # Remove ASCII-256 tokens
# tokens_not_in_top_10pct_filtered = tokens_not_in_top_10pct[tokens_not_in_top_10pct > 255]
# print(f"Number of these which not ASCII-256 = {len(tokens_not_in_top_10pct_filtered)}")

# # Inspect the remaining words (print the least copied out)
# str_toks_not_in_top_10pct_filtered = model.to_str_tokens(tokens_not_in_top_10pct_filtered)

# ranks = diag_elem_negative_ranks[tokens_not_in_top_10pct_filtered]
# ranks_ordered = torch.argsort(-ranks)
# ranks = ranks[ranks_ordered]
# str_toks_not_in_top_10pct_filtered = model.to_str_tokens(tokens_not_in_top_10pct_filtered[ranks_ordered])

# table = Table("Token", "Rank", title="OV circuit")
# for str_tok, rank in zip(str_toks_not_in_top_10pct_filtered, ranks[:30]):
#     table.add_row(str_tok, str(rank.item()))
# rprint(table)

In [ ]:
# # Attempt 1 for succession scores, keep this commented out, because it OOMs for Llama 3.2-1B
# task_succession_results = {}
# for task_name, task_prompt in task_prompts.items():
#     succession_score = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)
#     proportion_in_topk = {}
    
#     prompts = task_prompts[task_name]
#     tokens = model.to_tokens(prompts, prepend_bos=False)
#     total_tokens = tokens.shape[1]
#     for layer in tqdm(range(model.cfg.n_layers)):
#         for head in range(model.cfg.n_heads):
#             token_success_count = 0  # Count tokens where the condition is met
#             for token_idx in range(total_tokens - 1):  # Skip last token
                
#                 # Retrieve token embedding and successor ID
#                 token_id = tokens[0, token_idx].item()
#                 successor_id = tokens[0, token_idx + 1].item() 
#                 input_embed = W_EE[token_id]  # token_id, d_model

#                 # Calculate the effective OV circuit
#                 W_V = model.W_V[layer, head]
#                 W_O = model.W_O[layer, head]

#                 vec = input_embed @ model.W_V[layer, head]
#                 vec = vec @ model.W_O[layer, head]
#                 logits = vec @ W_U 

#                 # Take top-k logits and their indices
#                 _, topk_logits = torch.topk(logits, dim=-1, k=100) 

#                 # Find the position of the successor token in the top-k logits
#                 successor_pos_in_topk = None
#                 if successor_id in topk_logits:
#                     matches = (topk_logits == successor_id).nonzero(as_tuple=True)
#                     if matches[0].numel() > 0:  # Ensure there is at least one match
#                         successor_pos_in_topk = matches[0][0].item()  # Use the first match

#                 # Check positions of all other subsequent tokens in the sequence
#                 is_successor_before_others = True
#                 for other_idx in range(token_idx + 2, tokens.shape[1]):  # Loop through next tokens ignoring the first two (pos i and i+1)
#                     other_token_id = tokens[0, other_idx].item()

#                     # Check if the other token is in the top-k
#                     other_token_pos_in_topk = None
#                     if other_token_id in topk_logits:
#                         matches = (topk_logits == other_token_id).nonzero(as_tuple=True)
#                         if matches[0].numel() > 0:  # Ensure there is at least one match
#                             other_token_pos_in_topk = matches[0][0].item()  # Use the first match

#                     # If the successor is not before this token, fail the condition
#                     if successor_pos_in_topk is None or (other_token_pos_in_topk is not None and successor_pos_in_topk > other_token_pos_in_topk):
#                         is_successor_before_others = False
#                         break

#                 if is_successor_before_others:
#                     token_success_count += 1

#             # Compute the succession score
#             succession_condition_met = token_success_count > (total_tokens / 2)
#             proportion_in_topk[(layer, head)] = token_success_count / total_tokens
#             if succession_condition_met:
#                 succession_score[layer, head] = 1

#     task_succession_results[task_name] = {
#         "succession_scores": succession_score,
#         "proportion_in_topk": proportion_in_topk,
#     }
    

In [ ]:
# # Attempt 2 for succession scores, this doesn't OOM for Llama 3.2-1B, so we run this
def compute_succession_score(model: HookedTransformer, prompts: list[str], debug: bool = False) -> dict:
    succession_score = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)
    proportion_in_topk = {}
    succession_results = {}

    model.reset_hooks()
    
    act_cache = {}
    # Global Hook function
    def hook_fn(activation: torch.Tensor, hook: HookPoint, name: str = "activation"):
        """Stores activations in hook context."""
        act_cache[name] = activation
        return activation
    
    # Hook the activations of the effective circuit as described in the Successor Heads paper https://arxiv.org/pdf/2312.09230
    fwd_hooks = [
    ("hook_embed", partial(hook_fn, name="Embed")),
    ("blocks.0.hook_resid_pre", partial(hook_fn, name="Resid_Pre")),
    ("blocks.0.hook_mlp_out", partial(hook_fn, name="MLP0")), # not entirely sure the activation for MLP0, is it hook_resid_post or hook_mlp_out or mlp.hook_out
    ]
 
    model.run_with_hooks(prompts, fwd_hooks=fwd_hooks)

    # Retrieve activations
    WE_cached = act_cache["Embed"]  # Shape: [seq_len, d_model]
    # WE = model.embed  # Shape: [vocab_size, d_model]
    resid_pre = act_cache["Resid_Pre"]  # Shape: [batch, pos, d_model]
    MLP0 = act_cache["MLP0"]  # Shape: [batch, pos, d_model]
    WU = model.W_U  # Shape: [d_model, vocab_size]
    
    tokens = model.to_tokens(prompts, prepend_bos=False)
    total_tokens = tokens.shape[1]

    # Loop over all (layer, head) pairs
    for layer in tqdm(range(model.cfg.n_layers)):
        for head in range(model.cfg.n_heads):
            token_success_count = 0  # Count tokens where the condition is met
            for token_idx in range(total_tokens - 1):  # Skip last token
                
                # Retrieve token embedding and successor ID
                token_emb = WE_cached[:, token_idx]  # Shape: [d_model]
                successor_id = tokens[0, token_idx + 1].item()  # Successor token ID
                
                # # Compute the residual stream after MLP0
                MLP0_rearranged = einops.rearrange(MLP0, "b p d -> (b p) d")
                # MLP0_rearranged_single = MLP0_rearranged[token_idx, :]  # Shape: [d_model]
                resid_after_mlp1 = token_emb + MLP0
                
                # Compute W_OV and residual output
                W_OV = model.W_V[layer, head] @ model.W_O[layer, head]  
                assert W_OV.shape == (model.cfg.d_model, model.cfg.d_model)
                                
                pre_attention = model.blocks[0].ln1(resid_pre)
                attn_out = einops.einsum(
                    pre_attention, 
                    model.W_V[0],
                    model.W_O[0],
                    "b s d_model, num_heads d_model d_head, num_heads d_head d_model_out -> b s d_model_out",
                )
                resid_mid = attn_out + resid_pre
                normalized_resid_mid = model.blocks[0].ln2(resid_mid)
                mlp_out = model.blocks[0].mlp(normalized_resid_mid)
                
                W_EE = mlp_out.squeeze()
                W_EE_full = resid_mid.squeeze() + mlp_out.squeeze()

                effective_ov_circuit = W_EE @ W_OV @ WU

                # Compute effective circuit logits
                # logits = WU(ln_final(resid_after_ov)).squeeze()  # Shape: [vocab_size]

                # probs = torch.softmax(logits, dim=-1)
                # cumulative_probs = probs.sort(descending=True).values.cumsum(dim=-1)
                # k_dynamic = (cumulative_probs < 0.95).sum().item()  # k where 95% of the probability mass is included
                # k = min(k_dynamic, 100)  # Cap k at 100 for efficiency

                # Compute top-k logits and their indices
                _, topk_logits = torch.topk(effective_ov_circuit, dim=-1, k=20) 

                # Find the position of the successor token in the top-k logits
                successor_pos_in_topk = None
                if successor_id in topk_logits:
                    matches = (topk_logits == successor_id).nonzero(as_tuple=True)
                    if matches[0].numel() > 0:  # Ensure there is at least one match
                        successor_pos_in_topk = matches[0][0].item()  # Use the first match

                # Check positions of all other subsequent tokens in the sequence
                is_successor_before_others = True
                for other_idx in range(token_idx + 2, tokens.shape[1]):  # Loop through next tokens ignoring the first two (pos i and i+1)
                    other_token_id = tokens[0, other_idx].item()

                    # Check if the other token is in the top-k
                    other_token_pos_in_topk = None
                    if other_token_id in topk_logits:
                        matches = (topk_logits == other_token_id).nonzero(as_tuple=True)
                        if matches[0].numel() > 0:  # Ensure there is at least one match
                            other_token_pos_in_topk = matches[0][0].item()  # Use the first match

                    # If the successor is not before this token, fail the condition
                    if successor_pos_in_topk is None or (other_token_pos_in_topk is not None and successor_pos_in_topk > other_token_pos_in_topk):
                        is_successor_before_others = False
                        break

                if is_successor_before_others:
                    token_success_count += 1

                # Debugging Output
                if debug: 
                    current_token = model.tokenizer.decode([tokens[0, token_idx].item()], clean_up_tokenization_spaces=True).strip()
                    successor_token = model.tokenizer.decode([successor_id], clean_up_tokenization_spaces=True).strip()
                    print(f"Current Token: '{current_token}', Successor Token: '{successor_token}'")
                    print(f"Successor Position in Top-k: {successor_pos_in_topk}")
                    if not is_successor_before_others:
                        print("The successor appears after another token in the top-k.")

            # Compute the succession score
            succession_condition_met = token_success_count > (total_tokens / 2)
            proportion_in_topk[(layer, head)] = token_success_count / total_tokens
            if succession_condition_met:
                succession_score[layer, head] = 1

    succession_results = {
        "succession_scores": succession_score,
        "proportion_in_topk": proportion_in_topk,
    }

    return succession_results

task_succession_scores = {}

# for task, task_prompt in aug_data.items(): 
#     # Compute succession scores for this task
#     prompts = [task_prompt]  # Single task as input
#     task_succession_scores[task] = compute_succession_score(model, prompts)

# Compute succession scores for all tasks in the augmented data
for i, item in enumerate(aug_data):
    task_name = item["task"]
    task_prompt = item["clean"]
    task_succession_scores[task_name] = compute_succession_score(model, task_prompt, debug=False)

In [ ]:
def top_successors_heads(succession_results, i):
    top_values = sorted(succession_results['proportion_in_topk'].values(), reverse=True)[:i]
    top_indexes = [k for k, v in succession_results['proportion_in_topk'].items() if v in top_values]
    top_successors_heads = {}
    top_successors_heads["scores"] = top_values
    top_successors_heads["index"] = top_indexes
    for idx, score in zip(top_successors_heads["index"], top_successors_heads["scores"]):
        print(f"Head {idx}: {score:.4f}")

for task, task_result in task_succession_scores.items():
    print(f"Top 5 Successor Heads for Task: {task}")
    top_successors_heads(task_result, 5)

In [ ]:
from utils.plot_head import show_attention_patterns
model.cfg.ungroup_grouped_query_attention = False
# model.cfg.tokenizer_prepends_bos = False
# model.cfg.default_prepend_bos = False

numbers_prompt_aug = 'The last item in the sequence 0 1 2 3 4 5 6 7 8 9 20 21 22 is'
cardinals_prompt_aug = "The last item in the sequence first second third fourth fifth sixth seventh eighth ninth tenth is"
days_prompt_aug = 'The last item in the sequence monday tuesday wednesday thursday friday saturday sunday is'

show_attention_patterns(model, [(0, 21)], prompts=cardinals_prompt_aug, mode="ov", return_fig=True) 
show_attention_patterns(model, [(0, 21)], prompts=cardinals_prompt_aug, mode="pattern", return_fig=True) 

In [ ]:
mtx_ov = show_attention_patterns(model, [(11, 14)], prompts=task_prompts['Number words'], mode="ov", return_fig=False, return_mtx=True) 
mtx_qk = show_attention_patterns(model, [(11, 14)], prompts=task_prompts['Number words'], mode="pattern", return_fig=False, return_mtx=True) 

print("Single-head OV logits range:", mtx_ov.min().item(), mtx_ov.max().item(), mtx_ov.mean().item(), mtx_ov.std().item())
print("Single-head QK logits range:", mtx_qk.min().item(), mtx_qk.max().item(), mtx_qk.mean().item(), mtx_qk.std().item())

The QK attention pattern isn't showing a meaningful diagonal pattern below with one under the main diagonal similar to a Previous Token Head, meaning that claiming that a head is a Successor Head by looking at either the effective OV circuit or the QK circuit might not be sufficient for identifying succesive behavior. However, by inspecting either of them we shouldn't forget that it is due to the property of factorization of Attention that we are allowed to analyze them separately. They are both part of the Attention Head, such that the OV circuit isn't more part of the Successor Head than the QK circuit and vice-versa, because one can as easily disrupt the other by having near zero norm, making the output also small in norm. To investigate this possible effect one can take the pattern (post-softmax) and weight it by the L2-norm of the value vector (from this [paper](https://aclanthology.org/2020.emnlp-main.574.pdf)) element-wise.

However, in the case of observational analysis shortcuting the computation by isolating circuits seems thus to be an ad-hoc procedure in claiming properties to Attention Heads. In reality, the computation is much more complex and involves many more mechanisms than overlap into what is is said to be distributed behavior (i.e. because the Transformer is writing (and reading) into the residual stream a composition, albeit additive, of functions of the Attention and MLP). 

In [ ]:
model.cfg.ungroup_grouped_query_attention = True
for n, p in model.named_parameters(): 
    assert 'W_V' not in n, f"Found W_V in {n}"
    assert 'W_K' not in n, f"Found W_K in {n}"

## SVD Analysis of OV circuits

Resources: 

1. `SVDInterpreter` code breakdown: in this [LW post](https://www.lesswrong.com/posts/mkbGjzxD8d8XqKHzA/the-singular-value-decompositions-of-transformer-weight#Directly_editing_SVD_representations) & [colab](https://colab.research.google.com/drive/1G5e5I6zEKZUkuV6DR3yZEhiCoWCTLQdh?usp=sharing#scrollTo=EucG7uvy_z10)

2. Andrew Saxe's [PhD thesis](https://stacks.stanford.edu/file/nv482qj2831/Thesis-augmented.pdf):
    - Chapter 5: 
        - Sec. 5.3 "Acquiring Knowledge", Subsec. 5.3.1, Figure 5.2 (p. 108)
        - Sec. 5.4 "Organizing and Encoding Knowledge (p. 113)
        - Ap. A.5 "Learning dynamics with task-aligned input correlations" (p. 146)

3. A mathematical theory of semantic development in deep neural networks, [Saxe et al.](https://arxiv.org/pdf/1810.10531) (2018): 
    - Reuses the previous claims, maybe more explictly
    - Shows that under orthogonal assumption of the input, the singular values from the SVD decomposed input-output correlation matrices between the converge under sigmoidal like trajectories (start from 0, bcus of orthogonality, to $\infty$) see Figure 3 B,C and demonstration around Eq. 6 and Eq. 9 to a shallow network. This finding is also presented in Saxe's thesis (from Sec. 2.2.1 to 2.4, p. 43-53). Furthermore, these properties can be used to find weight initializations, but idk if it is relevant here.
    - TODO: extend if OV or QK factorized matrices under SVD decomposition find input-output "modes" (Fig 3.B/C): 
        - A “mode” is a joint direction in input and output space that explains a principal component of the correlation between them. It’s made up of:
            - A column of U: correlated properties
            - A row of 𝑉^⊤: correlated items
            - A singular value: strength of correlation

        - For OV:    
            - Right singular vectors → input directions in residual stream that the head is sensitive to.
            - Left singular vectors → output directions the head writes into.
            - Singular values → how strongly this mapping amplifies those directions.
            - If v1 is the top right singular vector, feeding an input aligned with v1 through the head will produce the **largest output**, aligned with u1 and scaled by s1.

        - For QK: 
            - Right singular vectors  -> source tokens directions giving attention
            - Left singular vectors -> destination tokens directions receiving attention
            - Singular values -> measures the strength of each interaction pattern, because the matrix is low-rank the an attention head uses a few directions that might contain a lot of collapsed information (e.g. universality of a attention head)
            - The top singular value tells you how strongly the top direction contributes.

        - In linear networks trained by gradient descent, the effective singular values grow monotonically over time.
        - This sigmoidal growth is a signature of stage-wise learning: the network first learns high-variance modes, then slowly fits lower-variance ones. This is directly tied to the power-law scaling of learning times shown in Eq. 6.
        - Find a connection to the OV/QK matrices from the previously identified successor heads in Llama3.2-1B 
            - Possible connection to Saxe et al.’s A(t) trajectory - the learned signal encoded in OV matrix, and ablating a singular vector is like interrupting the learning trajectory of a specific "mode". Some directions encode distracting semantics, and removing them can help prediction of successive tokens (e.g. “4”). Others encode useful inductive structure and ablating them may degrade prediction. By ablating a single singular vector (rank-1 update), we are essentially simulating targeted forgetting (for more info check out the [ROME](https://arxiv.org/pdf/2202.05262) paper on the misleading difference between *forgetting* and *editing*).


4. --//-- to a recent [study](https://arxiv.org/pdf/2406.09519) by Merullo et al. (2024) on inter-layer attention head communication. What happens if we ablate part of the communication channel between two heads (parametrized by QK/OV)? Sec. 4.1 and 5 seem to answer this, check it: "One way to think about this is zeroing out one singular value of e.g., the OV or matrix, or subtracting one of the component matrices from the sum in the equation: $W = \sum_{i=0}^{h} s_i \cdot U_i \otimes V_i$ (this intervention is shown further down)"



In [ ]:
from utils import SVDInterpreter
import pysvelte

svd_interpreter = SVDInterpreter(model)
ov = svd_interpreter.get_singular_vectors('OV', layer_index=11, head_index=14)

all_tokens = [model.to_str_tokens(np.array([i])) for i in range(model.cfg.d_vocab)]
all_tokens = [all_tokens[i][0] for i in range(model.cfg.d_vocab)]

def plot_matrix(matrix, tokens, k=40, filter="topk"):
    pysvelte.TopKTable(
        tokens=all_tokens,
        activations=matrix,
        obj_type="SVD direction",
        k=k,
        filter=filter
    ).show()

plot_matrix(ov, all_tokens)

In [ ]:
# qk = svd_interpreter.get_singular_vectors("QK", layer_index=11, head_index=14)
# plot_matrix(qk, all_tokens)

In [ ]:
def plot_singular_value_distribution(W_V_heads, W_O_heads,layer_idx, head_idx, max_rank=100):
  W_V_tmp, W_O_tmp = W_V_heads[layer_idx, head_idx, :], W_O_heads[layer_idx, head_idx]
  OV = W_V_tmp @ W_O_tmp
  U,S,V = torch.linalg.svd(OV)
  if max_rank > len(S):
    max_rank = len(S) -1
  plt.plot(S[0:max_rank].detach().cpu().numpy())
  # plt.yscale('log')
  plt.ylabel("Singular value")
  plt.xlabel("Rank")
  plt.title("Distribution of the singular vectors")
  plt.show()

# plot_singular_value_distribution(model.W_V, model.W_O, layer_idx = 11, head_idx = 14)
plot_singular_value_distribution(model.W_V, model.W_O, layer_idx = 11, head_idx = 14,max_rank = 64)

In [ ]:
def cosine_sim(x,y):
    return torch.dot(x,y) / (torch.norm(x) * torch.norm(y))

def svd_trace(model, tokens, layer_idx, N_singular_vectors = 20, sim_threshold = 0.15):
  emb = model.W_U
  token_list = model.tokenizer.encode(tokens)
  embed_dist = torch.zeros(emb.shape[1]).cuda()
  embed_dist[token_list] = 1 
  # project to embedding space
  embed_proj = embed_dist @ torch.linalg.pinv(emb)
  # normalize
  embed_proj = embed_proj / torch.sum(embed_proj)

  head_sims = []
  for n in range(model.cfg.n_heads):
      W_V_tmp, W_O_tmp = model.W_V[layer_idx, n], model.W_O[layer_idx, n]
      OV = W_V_tmp @ W_O_tmp
      U,S,V = torch.linalg.svd(OV)
      sims = []
      for i in range(N_singular_vectors):
          sim = cosine_sim(embed_proj, V[i,:]).item()
          if sim <=sim_threshold and sim >=-sim_threshold:
              sims.append(0)
          else:
              print("SIM FOUND: " + str(n) + " " + str(i) + " " + str(sim))
              sims.append(sim)
              emb_proj = V[i,:] @ emb
              topk, indices = emb_proj.topk(k=20)
              print("Found tokens:", model.tokenizer.decode(indices))
      sims = np.array(sims)
      head_sims.append(sims)
  head_sims = np.array(head_sims)
  plt.imshow(head_sims)
  plt.xticks(np.arange(0, N_singular_vectors))
  plt.yticks(np.arange(0, model.cfg.n_heads))
  plt.title("Similarities layer :" + str(layer_idx))
  plt.ylabel("Head number")
  plt.xlabel("Singular vector")
  plt.tight_layout()
  plt.show()
  return head_sims

tokens= 'both duo pair couple pair february second'

head_sims = svd_trace(model, tokens, layer_idx=11)

In [ ]:
import functools

# defining helper functions 
def rgetattr(obj, attr, *args):
    def _getattr(obj, attr):
        return getattr(obj, attr, *args)
    return functools.reduce(_getattr, [obj] + attr.split('.'))

def rsetattr(obj, attr, val):
    pre, _, post = attr.rpartition('.')
    return setattr(rgetattr(obj, pre) if pre else obj, post, val)

def single_rank_update(A ,idx):
    U,S,V = torch.linalg.svd(A)
    singval = S[idx]
    Vvec = V[idx,:]
    Uvec = U[:,idx]
    Uvec = Uvec.reshape(len(Uvec),1)
    Vvec = Vvec.reshape(len(Vvec),1)
    low_rank_update = singval * (Uvec @ Vvec.T)
    B = A - low_rank_update
    return B, low_rank_update

def subtract_singular_vector(OV, layer_idx, head_idx, update_idx, verbose = False):
    head_size = model.cfg.d_head
    hidden_dim = model.cfg.d_model 
    with torch.no_grad():
      new_OV = torch.tensor(OV, dtype=torch.float64)
      new_OV, _ = single_rank_update(new_OV,update_idx)
    # SVD and re-put together the new updatd matrix
      U, S, V = torch.linalg.svd(new_OV)
      Sdiag = torch.diag_embed(S)
      S64 = Sdiag[0:head_size, 0:head_size]
      S64sqrt = torch.sqrt(S64)
      M1 = U[:, 0:head_size] @ S64sqrt
      M2 =  S64sqrt @ V[0:head_size,:] 
      OV_redone = M1 @ M2
      if verbose:
        print("dist: ", torch.dist(OV_redone, new_OV))
      
    # Update W_O (output projection)
    W_O_path = f"blocks.{layer_idx}.attn.W_O"
    W_O = rgetattr(model, W_O_path).detach().clone()
    W_O[head_idx, :, :] = M2.to(W_O.dtype)  # Already [head_size, d_model]
    rgetattr(model, W_O_path).data.copy_(W_O)

    # Update W_V (value projection)
    W_V_path = f"blocks.{layer_idx}.attn.W_V"
    W_V = rgetattr(model, W_V_path).detach().clone()
    W_V[head_idx, :, :] = M1.to(W_V.dtype)  # Transpose to [head_size, d_model]
    rgetattr(model, W_V_path).data.copy_(W_V)


def removing_singular_vector_comparison(text, layer_idx, head_idx, N_singular_vectors = 10, verbose=False):
  tokens = model.tokenizer.encode(text, return_tensors='pt').cuda()
  logits = model.forward(tokens, return_type="logits")
  # decode output logits
  next_tokens_logits = logits[:, -1].clone()
  # get probs before applying filters
  original_next_tokens_probs = torch.nn.functional.softmax(next_tokens_logits, dim=-1)
  correct_idx = torch.argmax(original_next_tokens_probs)
  print("Predicted token: ", model.tokenizer.decode(correct_idx))
  if verbose:
    print(original_next_tokens_probs.shape)
    print(original_next_tokens_probs[0,correct_idx].log())
    print(model.tokenizer.decode(correct_idx))

  correct_logprob = original_next_tokens_probs[0,correct_idx].log()
  logprobs = []

  # get OV circuit
  W_V_tmp, W_O_tmp = model.W_V[layer_idx, head_idx, :], model.W_O[layer_idx, head_idx]
  OV = W_V_tmp @ W_O_tmp

  for i in range(N_singular_vectors):
      subtract_singular_vector(OV, layer_idx = layer_idx, head_idx = head_idx, update_idx = i)
      logits = model.forward(tokens, return_type="logits")
      # decode output logits
      next_tokens_logits = logits[:, -1].clone()
      # get probs before applying filters
      original_next_tokens_probs = torch.nn.functional.softmax(next_tokens_logits, dim=-1)
      correct_idx = torch.argmax(original_next_tokens_probs)
      correct_logprob = original_next_tokens_probs[0,correct_idx].log().item()
      if verbose:
        print("correct logprob: " + str(i) + "  " + str(correct_logprob))
      logprobs.append(correct_logprob)

  plt.plot(logprobs)
  plt.xlabel("Singular vector")
  plt.ylabel("Log probability")
  plt.title("Logprob when ablating singular vector")
  plt.tight_layout()
  plt.show()

  if verbose:
    return logprobs

### Editing SVD representations

$M = \sum_{i=1}^{\text{rank}(M)} S_i * U_i V_i^T$

Where $S_i$ is the ith singular vector, and $U_i$ and $V_i$ are the i'th columns of the left and right singular vector matrices. Given this sum, it is straightforward to see that we can calculate a similar matrix but without this singular vector with the rank one update.

$\hat{M} = M - S_i * U_i V_i^T$ ,

Where $i$ is the direction we want to intervene upon.

In [ ]:
ov_14 = model.W_V[11, 14, :] @ model.W_O[11, 14]
OV_updated, low_rank_update = single_rank_update(ov_14, 7)
U, S, V = torch.linalg.svd(OV_updated)
Vs = []
emb = model.W_U
N_singular_vectors = 10
for i in range(N_singular_vectors):
    acts = V[i,:].float() @ emb
    Vs.append(acts)
    
Vs = torch.stack(Vs, dim=1).unsqueeze(1)

# before removing the singular vector
plot_matrix(ov, all_tokens)
# after removing the singular vector
plot_matrix(Vs, all_tokens)

In [ ]:
# text = "Be careful leaving the iron on because it might catch"
text = "The day after the 3rd of July is the "

logprobs = removing_singular_vector_comparison(text, layer_idx=11, head_idx = 14, verbose=True)

In [ ]:
import math

print(f"Probabiliy of the predicting the correct token after removing singular vector 6: {4} is {(math.exp(-0.2435300648212433) * 100):.4f} %")
print(f"Probabiliy of the predicting the correct token after removing singular vector 7: {4} is {(math.exp(-0.2417430281639099) * 100):.4f} %")
print(f"Probabiliy of the predicting the correct token after removing singular vector 8: {4} is {(math.exp(-0.2641794681549072) * 100):.4f} %")